# Gate features

One row per slate (not per candidate). They feed the gate `w = sigmoid(a·log1p(history_length) + b·log1p(time_since_last_activity) + c·log1p(purchase_count) + bias)` that mixes Model 1 (short-term) and Model 2 (long-term): `score = w·P1 + (1-w)·P2`. For very short histories the gate is overridden to `w = 1` (Model 1 only).

**Inputs:** `data/train_sampled.parquet`, `val.parquet`, `test.parquet` (only the history columns are read).
**Outputs:** `features/gate_train.parquet`, `gate_val.parquet`, `gate_test.parquet`.

Same leakage rule as the Model 1 features: history events within `MIN_AGE_DAYS` of the query are dropped first, because they include the slate's own events and would inflate history length and purchase count and shrink the time since last activity.

## 0. Setup

In [1]:
import os
import numpy as np, pandas as pd, pyarrow.parquet as pq

DATA = "data"
OUT = "features"
MIN_AGE_DAYS = 0.01        # same value as the feature notebook
SHORT_HISTORY_PCTL = 10    # hard override w=1 below this train percentile of history length
os.makedirs(OUT, exist_ok=True)

HIST = ["slate_id", "history_type_list", "history_days_before"]


def load(path, columns=None):
    # ignore_metadata avoids the pandas 2.x/3.x pyarrow-metadata TypeError
    return pq.read_table(path, columns=columns).to_pandas(ignore_metadata=True)

## 1. Gate feature definitions
- `gate_history_length`: number of history events (all types) older than `MIN_AGE_DAYS`.
- `gate_time_since_last_activity`: days since the most recent such event. Empty history -> filled with the maximum observed age (365) and flagged.
- `gate_purchase_count`: number of ORD events in history. Orders are observed over the full year, so unlike clicks and carts this count is comparable across users.
- `log1p` versions are what enter the sigmoid (long right tails).

In [2]:
def build_gate_features(df, min_age=MIN_AGE_DAYS):
    n_rows = len(df)
    lens = df["history_type_list"].map(len).to_numpy()
    row = np.repeat(np.arange(n_rows), lens)
    typ = np.concatenate(df["history_type_list"].to_numpy()).astype(str)
    days = np.concatenate(df["history_days_before"].to_numpy()).astype(np.float64)
    keep = days >= min_age
    row, typ, days = row[keep], typ[keep], days[keep]

    length = np.bincount(row, minlength=n_rows)
    purchases = np.bincount(row[typ == "ORD"], minlength=n_rows)
    last = pd.Series(days).groupby(row).min().reindex(range(n_rows))       # NaN when no events left
    empty = length == 0
    last = last.fillna(365.0).to_numpy()

    out = pd.DataFrame({
        "slate_id": df["slate_id"].to_numpy(),
        "gate_history_length": length.astype(np.int32),
        "gate_time_since_last_activity": last.astype(np.float32),
        "gate_purchase_count": purchases.astype(np.int32),
        "gate_empty_history": empty.astype(np.int8),
    })
    out["gate_log_history_length"] = np.log1p(out["gate_history_length"]).astype(np.float32)
    out["gate_log_time_since_last"] = np.log1p(out["gate_time_since_last_activity"]).astype(np.float32)
    out["gate_log_purchase_count"] = np.log1p(out["gate_purchase_count"]).astype(np.float32)
    return out

## 2. Build for all splits

In [3]:
gate = {}
for name, fn in [("train", "train_sampled"), ("val", "val"), ("test", "test")]:
    d = load(f"{DATA}/{fn}.parquet", columns=HIST)
    gate[name] = build_gate_features(d)
    print(name, gate[name].shape)
    del d

train (139882, 8)
val (16924, 8)
test (17025, 8)


## 3. Short-history override threshold
The threshold is the 10th percentile of `gate_history_length` on **train only**, recomputed after the recent-event filter (the earlier value of 10 was computed before the filter).

In [4]:
thr = int(np.percentile(gate["train"]["gate_history_length"], SHORT_HISTORY_PCTL))
print("short-history threshold (train 10th percentile):", thr)
for name, g in gate.items():
    g["gate_short_history"] = (g["gate_history_length"] <= thr).astype(np.int8)
    g.to_parquet(f"{OUT}/gate_{name}.parquet", index=False)
    print(name, "override share:", round(g["gate_short_history"].mean() * 100, 2), "% | empty histories:", int(g["gate_empty_history"].sum()))

short-history threshold (train 10th percentile): 9
train override share: 10.9 % | empty histories: 1249
val override share: 10.71 % | empty histories: 133
test override share: 10.59 % | empty histories: 159


## 4. Descriptive statistics

In [5]:
cols = ["gate_history_length", "gate_time_since_last_activity", "gate_purchase_count"]
for name, g in gate.items():
    print("=====", name)
    print(g[cols].describe(percentiles=[.1, .25, .5, .75, .9, .99]).round(3).to_string())

===== train
       gate_history_length  gate_time_since_last_activity  gate_purchase_count
count           139882.000                     139882.000           139882.000
mean               150.301                         10.832               39.036
std                223.093                         42.263               70.064
min                  0.000                          0.010                0.000
10%                  9.000                          0.023                1.000
25%                 27.000                          0.209                4.000
50%                 76.000                          1.071               15.000
75%                184.000                          3.963               44.000
90%                371.000                         18.719              103.000
99%               1050.000                        311.423              327.000
max               4801.000                        365.000             2962.000
===== val
       gate_history_length  ga

## 5. Sanity checks

In [6]:
for name, g in gate.items():
    assert g["slate_id"].is_unique
    assert (g["gate_history_length"] >= 0).all() and (g["gate_purchase_count"] <= g["gate_history_length"]).all()
    assert g["gate_time_since_last_activity"].between(MIN_AGE_DAYS, 366).all()
    assert not g.drop(columns=[]).isna().any().any()
print("basic checks OK")

# brute-force check on a small sample (val)
d = load(f"{DATA}/val.parquet", columns=HIST).head(200)
g = gate["val"].set_index("slate_id")
for sid, types, days in zip(d["slate_id"], d["history_type_list"], d["history_days_before"]):
    k = [(t, x) for t, x in zip(types, days) if x >= MIN_AGE_DAYS]
    assert g.loc[sid, "gate_history_length"] == len(k)
    assert g.loc[sid, "gate_purchase_count"] == sum(t == "ORD" for t, _ in k)
    if k:
        assert abs(g.loc[sid, "gate_time_since_last_activity"] - min(x for _, x in k)) < 1e-4
print("brute-force check on 200 val slates OK")

# train vs val vs test distribution (they should look alike)
print(pd.DataFrame({n: g[["gate_log_history_length", "gate_log_time_since_last", "gate_log_purchase_count"]].mean()
                    for n, g in gate.items()}).round(3))

basic checks OK
brute-force check on 200 val slates OK
                          train    val   test
gate_log_history_length   4.200  4.234  4.203
gate_log_time_since_last  1.138  1.123  1.142
gate_log_purchase_count   2.686  2.714  2.693


## 6. Next steps
1. When Model 1 and Model 2 both produce scores, fit `a, b, c, bias` on **val** (full slates) to maximise ranking quality of `w·P1 + (1-w)·P2`, then report on test.
2. Before that, a useful evidence table: Model 1's NDCG@10 by gate bucket (history length, purchase count, time since last activity). If Model 1 is relatively better for short histories, the gate has a reason to exist.